<a href="https://colab.research.google.com/github/KapilFukte/Talking-Avatar/blob/main/talking_avatar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
%%bash
set -e
cd /content
[ -d SadTalker ] || git clone -q https://github.com/OpenTalker/SadTalker.git
if [ ! -d /content/miniconda3 ]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O mc.sh
  bash mc.sh -b -p /content/miniconda3
fi
source /content/miniconda3/bin/activate
[ -d /content/miniconda3/envs/sadtalker ] || conda create -y -q -n sadtalker -c conda-forge --override-channels python=3.8
conda activate sadtalker
cd SadTalker
pip install -q torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2
pip install -q -r requirements.txt
sed -i 's/torchvision.transforms.functional_tensor/torchvision.transforms.functional/' \
  /content/miniconda3/envs/sadtalker/lib/python3.8/site-packages/basicsr/data/degradations.py
echo "INSTALL DONE"


In [ ]:
%%bash
cd /content/SadTalker
mkdir -p checkpoints gfpgan/weights
B=https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc
for f in mapping_00109-model.pth.tar mapping_00229-model.pth.tar SadTalker_V0.0.2_256.safetensors SadTalker_V0.0.2_512.safetensors; do
  wget -q -nc $B/$f -P checkpoints
done
wget -q -nc https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth -P gfpgan/weights
wget -q -nc https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth -P gfpgan/weights
wget -q -nc https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth -P gfpgan/weights
wget -q -nc https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth -P gfpgan/weights
ls checkpoints gfpgan/weights && echo "MODELS DONE"


In [ ]:
from google.colab import drive
import os, glob, shutil, subprocess
drive.mount('/content/drive')
D = '/content/drive/MyDrive/TalkingAvatar'
os.makedirs('/content/in', exist_ok=True)

img = glob.glob(D + '/avatar.*')[0]
IMG = '/content/in/avatar' + os.path.splitext(img)[1].lower()
shutil.copy(img, IMG)

voice = glob.glob(D + '/voice.*')[0]
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', voice, '-ar', '16000', '-ac', '1', '/content/in/voice.wav'], check=True)
print('Ready:', img, voice)


In [ ]:
# ===== SETTINGS (change these) =====
MODE = "head"      # "head" = face close-up, natural blinks + small head movement
                   # "full" = your whole image, body stays still
EXPRESSION = 1.0   # 0.8 = calmer, 1.0 = normal, 1.3 = more expressive
SHARPEN = True     # face sharpening (better quality, a bit slower)
# ===================================

extra = "--preprocess crop" if MODE == "head" else \
        "--preprocess full --still --ref_eyeblink examples/ref_video/WDA_KatieHill_000.mp4"
if SHARPEN: extra += " --enhancer gfpgan"

cmd = (f"source /content/miniconda3/bin/activate sadtalker && cd /content/SadTalker && "
       f"python inference.py --driven_audio /content/in/voice.wav --source_image {IMG} "
       f"--result_dir /content/out --size 512 --expression_scale {EXPRESSION} {extra}")
!bash -c "{cmd}"


In [ ]:
import glob, os, shutil, datetime
vids = sorted(glob.glob('/content/out/**/*.mp4', recursive=True), key=os.path.getmtime)
name = 'video_' + datetime.datetime.now().strftime('%Y%m%d_%H%M') + '.mp4'
shutil.copy(vids[-1], f'/content/drive/MyDrive/TalkingAvatar/{name}')
print('Saved to Google Drive → TalkingAvatar/' + name)
